# 01 — Extraction de texte

Extraction de texte depuis PDF (`pdfplumber`) et DOCX (`python-docx`), nettoyage
(en-têtes/pieds de page répétés, espaces multiples, encodage), puis export vers
`artifacts/documents_extraits.csv` (colonnes : `id`, `texte_brut`, `texte_nettoye`,
`nb_mots`, `label`).

La colonne `label` (catégorie du document) n'est pas demandée par le cahier des
charges initial, mais elle est nécessaire pour entraîner le classifieur du
notebook 2 (StratifiedKFold a besoin d'étiquettes) — elle est donc générée ici
en même temps que les documents de démonstration.

**Jeu de données** : en l'absence d'un corpus réel de documents 2M à ce stade,
ce notebook génère un petit corpus de démonstration réaliste (procédures,
contrats, notes RH, fiches techniques, briefs éditoriaux) — une partie sous
forme de vrais fichiers PDF/DOCX (pour valider l'extraction), une partie déjà
"digitalisée" (pour avoir un volume suffisant à classifier). À remplacer par
le vrai corpus documentaire 2M en production.

In [1]:
# Imports
import os, re, sys, random
from pathlib import Path
from collections import defaultdict
import pandas as pd
from tqdm import tqdm

import pdfplumber
import docx

from utils import ARTIFACTS_DIR, save_dataframe

SAMPLE_DIR = Path(ARTIFACTS_DIR) / "sample_docs"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)
print("Dossier des artefacts :", ARTIFACTS_DIR)
print("Dossier des documents de démo :", SAMPLE_DIR)

Dossier des artefacts : /home/claude/get-2m/ged-ai/artifacts
Dossier des documents de démo : /home/claude/get-2m/ged-ai/artifacts/sample_docs


## 1. Fonctions d'extraction et de nettoyage

In [2]:
def extract_text_pdf(path: str) -> str:
    text_parts = []
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text_parts.append(page.extract_text() or "")
    except Exception as e:
        print(f"Erreur lecture PDF {path}: {e}")
    return "\n".join(text_parts)


def extract_text_docx(path: str) -> str:
    text_parts = []
    try:
        d = docx.Document(path)
        for para in d.paragraphs:
            text_parts.append(para.text)
    except Exception as e:
        print(f"Erreur lecture DOCX {path}: {e}")
    return "\n".join(text_parts)


def clean_text(text: str) -> str:
    """Nettoyage : normalisation des espaces, suppression des lignes
    répétées (en-têtes/pieds de page probables), correction d'encodage."""
    if not isinstance(text, str):
        return ""
    t = text.replace("\r", "\n")
    t = re.sub(r"\n{2,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)

    lines = [ln.strip() for ln in t.splitlines() if ln.strip()]
    freq = defaultdict(int)
    for ln in lines:
        freq[ln] += 1
    # une ligne courte répétée plus de 3 fois est probablement un en-tête/pied de page
    filtered = [ln for ln in lines if not (freq[ln] > 3 and len(ln) < 80)]

    out = "\n".join(filtered)
    out = out.encode("utf-8", "replace").decode("utf-8")
    return out.strip()

## 2. Génération d'un corpus de démonstration

Deux sources, combinées dans le même CSV de sortie :

1. **Fichiers réels PDF/DOCX** générés puis passés dans le pipeline d'extraction
   ci-dessus (valide que `pdfplumber` et `python-docx` fonctionnent réellement).
2. **Documents synthétiques déjà "extraits"** (texte généré directement), pour
   avoir un volume suffisant par catégorie en vue du notebook 2 (classification).

In [3]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

CATEGORIES = {
    "Procedures_internes": [
        "Cette procédure décrit les étapes à suivre en cas d'incident technique sur le réseau de diffusion.",
        "Consigne de sécurité : évacuation immédiate du bâtiment en cas d'alerte incendie, point de rassemblement au parking nord.",
        "Procédure de sauvegarde quotidienne des serveurs de diffusion, exécutée chaque nuit à 02h00.",
        "En cas de coupure d'antenne, contacter immédiatement l'astreinte technique au numéro interne dédié.",
        "Protocole de contrôle qualité avant diffusion : vérification du format, du son et des sous-titres.",
        "Procédure d'habilitation des nouveaux collaborateurs aux outils de régie et de montage.",
    ],
    "Contrats": [
        "Le présent contrat de prestation technique lie SOREAD-2M au prestataire pour la maintenance des équipements de diffusion.",
        "Accord de partenariat publicitaire entre 2M et l'annonceur, portant sur une campagne de trois mois.",
        "Contrat de licence pour l'utilisation du logiciel de montage vidéo, renouvelable annuellement.",
        "Convention de coproduction audiovisuelle entre 2M et le producteur externe pour la nouvelle émission.",
        "Contrat de prestation de services informatiques pour l'infogérance du parc serveurs.",
        "Avenant au contrat cadre fixant les nouvelles conditions tarifaires pour l'année en cours.",
    ],
    "RH": [
        "Note de service concernant la mise à jour du règlement intérieur applicable à tous les départements.",
        "Fiche de poste pour le recrutement d'un ingénieur systèmes au sein du département IT.",
        "Compte-rendu de l'entretien annuel d'évaluation, incluant les objectifs pour l'année suivante.",
        "Politique de télétravail applicable aux équipes techniques et administratives de 2M.",
        "Note d'information sur les congés payés et les modalités de pose pendant la période estivale.",
        "Procédure d'intégration des nouveaux employés durant leur première semaine.",
    ],
    "Technique": [
        "Fiche technique décrivant l'architecture du système de diffusion satellite utilisé par la chaîne.",
        "Documentation d'installation du nouveau serveur de stockage pour les archives vidéo.",
        "Rapport de test de la nouvelle plateforme de gestion électronique de documents.",
        "Spécifications techniques du système d'encodage vidéo haute définition.",
        "Guide de configuration réseau pour les postes de travail du département IT.",
        "Note technique sur la migration de la base de données vers PostgreSQL.",
    ],
    "Editorial": [
        "Brief éditorial pour le prochain numéro du journal télévisé du soir.",
        "Conducteur d'émission détaillant le déroulé minute par minute du programme.",
        "Note de cadrage pour le reportage sur l'agriculture dans la région du Souss.",
        "Synopsis de la nouvelle émission culturelle prévue pour la grille de rentrée.",
        "Compte-rendu de la conférence de rédaction hebdomadaire.",
        "Charte éditoriale rappelant les principes de neutralité et de vérification des sources.",
    ],
}

records = []

# --- 2a. Vrais fichiers PDF/DOCX pour valider l'extraction ---
sample_cats = ["Procedures_internes", "Contrats", "RH", "Technique"]
for i, cat in enumerate(sample_cats):
    text = CATEGORIES[cat][0] + " " + CATEGORIES[cat][1]

    pdf_path = SAMPLE_DIR / f"{cat}_sample.pdf"
    c = canvas.Canvas(str(pdf_path), pagesize=A4)
    c.setFont("Helvetica", 11)
    textobject = c.beginText(50, 800)
    for line in [text[j:j+90] for j in range(0, len(text), 90)]:
        textobject.textLine(line)
    c.drawText(textobject)
    c.save()

    docx_path = SAMPLE_DIR / f"{cat}_sample.docx"
    d = docx.Document()
    d.add_heading(cat.replace("_", " "), level=1)
    d.add_paragraph(text)
    d.save(str(docx_path))

    for path, kind in [(pdf_path, "pdf"), (docx_path, "docx")]:
        raw = extract_text_pdf(str(path)) if kind == "pdf" else extract_text_docx(str(path))
        clean = clean_text(raw)
        records.append({
            "id": path.name, "texte_brut": raw, "texte_nettoye": clean,
            "nb_mots": len(clean.split()), "label": cat,
        })

print(f"{len(records)} documents extraits depuis de vrais fichiers PDF/DOCX.")

8 documents extraits depuis de vrais fichiers PDF/DOCX.


In [4]:
# --- 2b. Documents synthétiques (déjà "digitalisés") pour compléter le volume ---
random.seed(7)
n_per_cat = 8
for cat, phrases in CATEGORIES.items():
    for k in range(n_per_cat):
        chosen = random.sample(phrases, k=min(3, len(phrases)))
        text = " ".join(chosen)
        clean = clean_text(text)
        records.append({
            "id": f"{cat}_synth_{k:02d}.txt",
            "texte_brut": text,
            "texte_nettoye": clean,
            "nb_mots": len(clean.split()),
            "label": cat,
        })

df = pd.DataFrame(records)
print("Total documents :", len(df))
print(df["label"].value_counts())

Total documents : 48
label
Procedures_internes    10
Contrats               10
RH                     10
Technique              10
Editorial               8
Name: count, dtype: int64


In [5]:
out_path = save_dataframe(df, "documents_extraits.csv")
print("Fichier sauvegardé :", out_path)
df.head(5)

Fichier sauvegardé : /home/claude/get-2m/ged-ai/artifacts/documents_extraits.csv


,id,texte_brut,texte_nettoye,nb_mots,label
0,Procedures_internes_sample.pdf,Cette procédure décrit les étapes à suivre en ...,Cette procédure décrit les étapes à suivre en ...,35,Procedures_internes
1,Procedures_internes_sample.docx,Procedures internes\nCette procédure décrit le...,Procedures internes\nCette procédure décrit le...,36,Procedures_internes
2,Contrats_sample.pdf,Le présent contrat de prestation technique lie...,Le présent contrat de prestation technique lie...,34,Contrats
3,Contrats_sample.docx,Contrats\nLe présent contrat de prestation tec...,Contrats\nLe présent contrat de prestation tec...,33,Contrats
4,RH_sample.pdf,Note de service concernant la mise à jour du r...,Note de service concernant la mise à jour du r...,32,RH


## Notes

- Les 8 premiers documents (2 PDF + 2 DOCX déjà réels, sur 4 catégories) prouvent
  que l'extraction `pdfplumber`/`python-docx` fonctionne réellement.
- Le reste du corpus est généré directement en texte pour disposer d'un volume
  suffisant par catégorie (`n_per_cat` réglable) — **à remplacer par le vrai
  corpus documentaire 2M dès qu'il est disponible**, sans changer le format de
  sortie (`documents_extraits.csv`).
- La colonne `label` est propagée pour le notebook 2. Si le vrai corpus n'a pas
  d'étiquettes, prévoir une phase d'annotation manuelle ou semi-automatique
  avant l'entraînement du classifieur.